In [ ]:
pip install pandas tqdm

In [ ]:
import pandas as pd

# Load Excel file
df = pd.read_excel("DhoroniDataset.xlsx")

print("Total articles:", len(df))
print(df.head())

Total articles: 2300
                    dates                                              Title  \
0     18 April 2021 11:49                      ঢাকায় বায়ু দূষণ রোধে করণীয়   
1  26 November 2019 04:34  বায়ু দূষণ কমাতে ঢাকাসহ ৫ জেলায় ইটভাটা বন্ধের...   
2  20 February 2024 08:07  ঝুঁকিপূর্ণ বাতাস: ঢাকায় ঘরের বাইরে মাস্ক পরার...   
3      08 June 2022 18:22  পরিবেশের স্বাস্থ্যরক্ষায় সব শেষে ভারত! অনুমানে...   
4      06 June 2022 08:32     পরিবেশ রক্ষায় বহুমাত্রিক প্রচেষ্টা, দাবি মোদীর   

          File Political Influence Scientific Data Mentioned  \
0     ID-1.txt                  No                       Yes   
1    ID-10.txt                  No                        No   
2   ID-100.txt                  No                        No   
3  ID-1000.txt                  No                        No   
4  ID-1003.txt                 Yes                        No   

  Statistical Data Mentioned Source Mentioned Authenticity Stance Detection  \
0                        Yes      

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset from Colab local storage
df = pd.read_excel("DhoroniDataset.xlsx")  # make sure the file is in Colab root

# Split into train (80%) and test (20%)
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df['label'] if 'label' in df.columns else None  # only if label exists
)

# Save the splits locally in Colab
train_df.to_excel("DhoroniDataset_train.xlsx", index=False)
test_df.to_excel("DhoroniDataset_test.xlsx", index=False)

# Print the shapes to confirm
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (1840, 13)
Test shape: (460, 13)


In [ ]:
import pandas as pd
from google.colab import files

# Step 1 — Load the existing train and test datasets
train_df = pd.read_excel("DhoroniDataset_train.xlsx")
test_df = pd.read_excel("DhoroniDataset_test.xlsx")

print("Train columns before dropping:", train_df.columns.tolist())
print("Test columns before dropping:", test_df.columns.tolist())

# Step 2 — Drop unwanted columns
columns_to_drop = [
    "dates",
    "Political Influence",
    "Scientific Data Mentioned",
    "Statistical Data Mentioned",
    "Location",
    "Target",
    "Authority Involvement"
]

train_df = train_df.drop(columns=[col for col in columns_to_drop if col in train_df.columns])
test_df = test_df.drop(columns=[col for col in columns_to_drop if col in test_df.columns])

print("Train columns after dropping:", train_df.columns.tolist())
print("Test columns after dropping:", test_df.columns.tolist())

# Step 3 — Save the cleaned datasets
train_file_cleaned = "DhoroniDataset_train_cleaned.xlsx"
test_file_cleaned = "DhoroniDataset_test_cleaned.xlsx"

train_df.to_excel(train_file_cleaned, index=False)
test_df.to_excel(test_file_cleaned, index=False)

print(f"Cleaned train dataset saved as {train_file_cleaned}")
print(f"Cleaned test dataset saved as {test_file_cleaned}")

# Step 4 — Download the cleaned files
files.download(train_file_cleaned)
files.download(test_file_cleaned)


Train columns before dropping: ['dates', 'Title', 'File', 'Political Influence', 'Scientific Data Mentioned', 'Statistical Data Mentioned', 'Source Mentioned', 'Authenticity', 'Stance Detection', 'Location', 'Focus', 'Target', 'Authority Involvement']
Test columns before dropping: ['dates', 'Title', 'File', 'Political Influence', 'Scientific Data Mentioned', 'Statistical Data Mentioned', 'Source Mentioned', 'Authenticity', 'Stance Detection', 'Location', 'Focus', 'Target', 'Authority Involvement']
Train columns after dropping: ['Title', 'File', 'Source Mentioned', 'Authenticity', 'Stance Detection', 'Focus']
Test columns after dropping: ['Title', 'File', 'Source Mentioned', 'Authenticity', 'Stance Detection', 'Focus']
Cleaned train dataset saved as DhoroniDataset_train_cleaned.xlsx
Cleaned test dataset saved as DhoroniDataset_test_cleaned.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

# Step 1 — Load the already cleaned train and test datasets
train_df = pd.read_excel("DhoroniDataset_train_cleaned.xlsx")
test_df = pd.read_excel("DhoroniDataset_test_cleaned.xlsx")

# Function to merge Title + File into text and keep only id, Focus, text
def merge_text(df):
    # Merge Title + File into a single 'text' column
    df['text'] = "Title: " + df['Title'].astype(str) + "\nArticle: " + df['File'].astype(str)

    # Generate id if missing
    if 'id' not in df.columns:
        df['id'] = range(1, len(df)+1)

    # Keep only id, Focus, text
    df = df[['id', 'Focus', 'text']]
    return df

# Step 2 — Apply function to both datasets
train_df_merged = merge_text(train_df)
test_df_merged = merge_text(test_df)

# Step 3 — Save merged train and test datasets
train_file_merged = "DhoroniDataset_train_final.xlsx"
test_file_merged = "DhoroniDataset_test_final.xlsx"

train_df_merged.to_excel(train_file_merged, index=False)
test_df_merged.to_excel(test_file_merged, index=False)

print(f"Final train dataset saved as {train_file_merged} (shape: {train_df_merged.shape})")
print(f"Final test dataset saved as {test_file_merged} (shape: {test_df_merged.shape})")

# Step 4 — Download the merged datasets
files.download(train_file_merged)
files.download(test_file_merged)


Final train dataset saved as DhoroniDataset_train_final.xlsx (shape: (1840, 3))
Final test dataset saved as DhoroniDataset_test_final.xlsx (shape: (460, 3))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Convert  to JSONL**

In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from google.colab import files

# Step 1 — Load cleaned train and test datasets
train_df = pd.read_excel("DhoroniDataset_train_final.xlsx")
test_df = pd.read_excel("DhoroniDataset_test_final.xlsx")

# Function to convert a dataframe to JSONL
def df_to_jsonl(df, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            record = {
                "id": row["id"],
                "focus": row["Focus"],  # keep focus label
                "text": row["text"],
                "labels": []  # initially empty; will be filled later
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"JSONL file saved as {output_file}")
    files.download(output_file)

# Step 2 — Convert train dataset to JSONL
df_to_jsonl(train_df, "DhoroniDataset_train.jsonl")

# Step 3 — Convert test dataset to JSONL
df_to_jsonl(test_df, "DhoroniDataset_test.jsonl")


100%|██████████| 1840/1840 [00:00<00:00, 13200.58it/s]

JSONL file saved as DhoroniDataset_train.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

100%|██████████| 460/460 [00:00<00:00, 13857.90it/s]

JSONL file saved as DhoroniDataset_test.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import json

# Paths to your train and test JSONL files
train_jsonl = "DhoroniDataset_train.jsonl"
test_jsonl = "DhoroniDataset_test.jsonl"

# Function to load JSONL into a list of dictionaries
def load_jsonl(jsonl_file):
    articles = []
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            articles.append(json.loads(line))
    print(f"Total articles loaded from {jsonl_file}: {len(articles)}")
    if len(articles) > 0:
        print("First article:")
        print(json.dumps(articles[0], ensure_ascii=False, indent=2))
    return articles

# Load train and test JSONL files
train_articles = load_jsonl(train_jsonl)
test_articles = load_jsonl(test_jsonl)


Total articles loaded from DhoroniDataset_train.jsonl: 1840
First article:
{
  "id": 1,
  "focus": "Plastic Pollution",
  "text": "Title: প্রশাসক বদল, ধুঁকছে প্লাস্টিক ধরার অভিযান\nArticle: ID-2048.txt",
  "labels": []
}
Total articles loaded from DhoroniDataset_test.jsonl: 460
First article:
{
  "id": 1,
  "focus": "Soil Pollution",
  "text": "Title: সভ্যতার হুমকি মাটি দূষণ\nArticle: ID-2943.txt",
  "labels": []
}


In [ ]:
import os

# Change this to your desired folder in Drive
drive_folder = "/content/drive/MyDrive/softcomproject/"

# Create folder if it doesn't exist
os.makedirs(drive_folder, exist_ok=True)


In [ ]:
import os
import shutil

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define your destination folder in Drive
drive_folder = '/content/drive/MyDrive/softcomproject'
os.makedirs(drive_folder, exist_ok=True)  # Create the folder if it doesn't exist

# List of files to move
files_to_move = [
    "DhoroniDataset.xlsx",
    "DhoroniDataset_cleaned.xlsx",
    "DhoroniDataset_test.jsonl",
    "DhoroniDataset_test.xlsx",
    "DhoroniDataset_test_cleaned.xlsx",
    "DhoroniDataset_test_final.xlsx",
    "DhoroniDataset_train.jsonl",
    "DhoroniDataset_train.xlsx",
    "DhoroniDataset_train_cleaned.xlsx",
    "DhoroniDataset_train_final.xlsx"
]

# Move files
for file_name in files_to_move:
    if os.path.exists(file_name):
        shutil.move(file_name, os.path.join(drive_folder, file_name))
    else:
        print(f"File not found: {file_name}")

print("✅ All available files moved to Drive folder:", drive_folder)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All available files moved to Drive folder: /content/drive/MyDrive/DhoroniFiles
